# 37 · Skill + hooks + permissions —— 让 Claude Code 「不弹窗」

> **学习目标**：理解 Claude Code 的 `settings.json` 里 3 个关键字段 —— `permissions.allow`（自动放行）、`hooks`（tool 前后插脚本）、环境变量注入。然后写一个自动化脚本让「装完一个 Skill 自动配好 settings」。
>
> **预备**：33-36 跑过。
>
> **为什么重要**：Skill 装了，但**每次调 Skill 都要点「Yes I trust」权限** = 没装。**hooks + permissions 让 Skill 真用起来**。

In [ ]:
import os, shutil, json, sys
from pathlib import Path
from typing import Callable
SBX = Path('./_skill_sandbox').resolve()
if SBX.exists(): shutil.rmtree(SBX)
SBX.mkdir()
HOME = SBX / 'home'
CLAUDE_DIR = HOME / '.claude'
CLAUDE_DIR.mkdir(parents=True)
print('沙箱就绪')

## 1. settings.json 三件套结构

**Claude Code 启动时加载** `~/.claude/settings.json`，含三大段：
1. **`permissions`** —— `allow` / `deny` / `ask` 三类规则（按工具名 + 命令前缀）
2. **`hooks`** —— tool 调用前/后 / 通知触发 shell 脚本（自动日志、自动 lint 之类）
3. **环境变量 / 模型配置** —— `env` / `model` 字段

我们写一个 `SettingsEditor` 类模拟「管理 settings.json」的能力。

In [ ]:
class SettingsEditor:
    def __init__(self, path: Path):
        self.path = path
        if path.exists():
            self.cfg = json.loads(path.read_text(encoding='utf-8'))
        else:
            self.cfg = {'permissions': {'allow': [], 'deny': []}, 'hooks': {}, 'env': {}}

    def allow_command(self, cmd_prefix: str):
        bucket = self.cfg.setdefault('permissions', {}).setdefault('allow', [])
        if cmd_prefix not in bucket:
            bucket.append(cmd_prefix)
            self.save()
            return f'Added allow: {cmd_prefix}'
        return f'Already allowed: {cmd_prefix}'

    def add_hook(self, event: str, matcher: str, command: str):
        """event: 'PreToolUse' | 'PostToolUse' | 'Notification'
           matcher: tool name or '*'
        """
        hooks = self.cfg.setdefault('hooks', {}).setdefault(event, [])
        hooks.append({'matcher': matcher, 'hooks': [{'type': 'command', 'command': command}]})
        self.save()
        return f'Added {event} hook for {matcher}'

    def set_env(self, key: str, value: str):
        self.cfg.setdefault('env', {})[key] = value
        self.save()

    def save(self):
        self.path.parent.mkdir(parents=True, exist_ok=True)
        self.path.write_text(json.dumps(self.cfg, ensure_ascii=False, indent=2), encoding='utf-8')

    def __repr__(self):
        return json.dumps(self.cfg, ensure_ascii=False, indent=2)

settings = SettingsEditor(CLAUDE_DIR / 'settings.json')
print('初始 settings.json:')
print(repr(settings))

## 2. 装 Skill 时自动配 permissions

**问题**：用户装了 `/commit-pr` Skill，跑起来后 `git push` / `gh pr create` **都要权限弹窗**。
**修法**：Skill 本身声明它需要什么命令，**安装器**自动加到 `permissions.allow`。

In [ ]:
# Skill 的 SKILL.md 加个新字段：需要自动放行的命令
commit_pr = CLAUDE_DIR / 'skills' / 'commit-pr' / 'SKILL.md'
commit_pr.parent.mkdir(parents=True)
commit_pr.write_text('''---
name: commit-pr
description: commit + open PR
permissions:
  - Bash(git add*)
  - Bash(git commit*)
  - Bash(git push*)
  - Bash(gh pr create*)
---

# Steps
1. git status, git diff
2. craft commit + PR
3. git push + gh pr create
''', encoding='utf-8')
print('已建 commit-pr Skill（带 permissions 字段）')

In [ ]:
# 解析 SKILL.md 的 permissions 字段，批量加进 settings
import re, yaml

def extract_permissions(skill_md: Path) -> list[str]:
    text = skill_md.read_text(encoding='utf-8')
    if not text.startswith('---'):
        return []
    end = text.find('\n---', 3)
    if end == -1: return []
    fm = yaml.safe_load(text[3:end]) or {}
    return fm.get('permissions', [])

def install_skill(skill_md: Path, settings: SettingsEditor, verbose: bool = True):
    perms = extract_permissions(skill_md)
    results = []
    for p in perms:
        m = re.match(r'(\w+)\((.+?)\)', p)
        if not m:
            results.append(f'  跳过格式不对的权限: {p}')
            continue
        tool, prefix = m.group(1), m.group(2)
        if tool == 'Bash':
            results.append(settings.allow_command(f'Bash({prefix})'))
    if verbose:
        print(f'\n装 Skill {skill_md.parent.name}:')
        for r in results:
            print(f'  {r}')
    return results

install_skill(commit_pr, settings)

In [ ]:
# 现在 settings.json 已经有 4 个允许规则了
print(repr(settings))
print()
print('→ 这 4 个命令以后跑就不再弹窗了。')

## 3. hooks：tool 前后自动跑脚本

**典型用法**：
- `PostToolUse` 跑 `Bash` 后 → 自动把命令结果写到 log
- `PreToolUse` 跑 `Write` 前 → 自动 git diff 备份
- `Notification` Claude 等你回复时 → 发系统通知

我们写一个「Bash 命令 → 自动写日志」的 hook 演示。

In [ ]:
# 写一个 logger 脚本（这是 hook 真正要执行的命令）
logger_script = CLAUDE_DIR / 'log_bash.py'
logger_script.write_text('''#!/usr/bin/env python
"""PostToolUse hook for Bash: log every command to ~/.claude/logs/bash.log"""
import sys, json, datetime
from pathlib import Path

log_file = Path(__file__).parent / 'logs' / 'bash.log'
    log_file.parent.mkdir(exist_ok=True)
    try:
        payload = json.loads(sys.stdin.read())
        tool_input = payload.get('tool_input', {})
        cmd = tool_input.get('command', '?')
        tool_response = payload.get('tool_response', '')
        with log_file.open('a', encoding='utf-8') as f:
            f.write(f'[{datetime.datetime.now().isoformat()}] $ {cmd}\n')
            f.write(f'    response: {str(tool_response)[:200]}\n')
        print('logged')
    except Exception as e:
        print(f'log error: {e}', file=sys.stderr)
        sys.exit(1)
    ''', encoding='utf-8')
print(f'已写 hook 脚本: {logger_script.name}')

In [ ]:
# 注册 hook：Bash 工具调用完后跑 logger_script
msg = settings.add_hook(
    event='PostToolUse',
    matcher='Bash',
    command=f'python {logger_script}'
)
print('注册 hook:', msg)
print()
print(repr(settings))

In [ ]:
# 模拟 Bash 工具调用 → hook 触发
import json as _json
import datetime as _dt
import subprocess as _sp

def simulate_bash_then_hook(cmd: str):
    payload = _json.dumps({
        'tool_input': {'command': cmd},
        'tool_response': f'(mock response for: {cmd})',
    })
    # 实际 Claude Code 会用这个 payload 通过 stdin 喂给 hook 脚本
    # 我们直接调脚本模拟
    result = _sp.run(
        ['python', str(logger_script)],
        input=payload, capture_output=True, text=True, encoding='utf-8'
    )
    return result.stdout.strip()

# 跑 3 次
for cmd in ['git status', 'ls -la /tmp', 'echo "hello hook"']:
    out = simulate_bash_then_hook(cmd)
    print(f'  hook result: {out}')

# 看 log 文件
log_file = CLAUDE_DIR / 'logs' / 'bash.log'
if log_file.exists():
    print(f'\n--- {log_file.name} ---')
    print(log_file.read_text(encoding='utf-8'))

## 4. 完整「装 Skill 一键配置」脚本

实战：写一个 `install_skill_full(skill_md)`，**一次性**干三件事：
1. 解析 `permissions` 字段 → 加进 `settings.json` 的 allow
2. 装 hook 脚本（如有）→ 加进 `settings.json` 的 hooks
3. 复制 Skill 目录到 `~/.claude/skills/`

In [ ]:
def install_skill_full(skill_md: Path, settings: SettingsEditor, dest_skills: Path, hook_scripts: dict | None = None):
    """完整的 Skill 装入流程。"""
    name = skill_md.parent.name
    print(f'\n=== 装 {name} ===')
    # 1. permissions
    perms = extract_permissions(skill_md)
    if perms:
        print(f'  [1/3] permissions: {len(perms)} 条')
        for p in perms: install_skill(skill_md, settings, verbose=False)
    # 2. hooks
    text = skill_md.read_text(encoding='utf-8')
    if '---' in text:
        end = text.find('\n---', 3)
        fm = yaml.safe_load(text[3:end]) or {}
        hooks = fm.get('hooks', [])
        if hooks:
            print(f'  [2/3] hooks: {len(hooks)} 条')
            for h in hooks:
                settings.add_hook(h.get('event', 'PostToolUse'), h.get('matcher', '*'), h.get('command', ''))
    # 3. 复制 skill 目录
    dest = dest_skills / name
    if dest.exists(): shutil.rmtree(dest)
    shutil.copytree(skill_md.parent, dest)
    print(f'  [3/3] 复制到 {dest}')
    print(f'  {name} 装入完成')

settings2 = SettingsEditor(CLAUDE_DIR / 'settings.json')
skills_root = CLAUDE_DIR / 'skills'
skills_root.mkdir(exist_ok=True)
install_skill_full(commit_pr, settings2, skills_root)
print()
print('最终 settings:')
print(repr(settings2))

In [ ]:
shutil.rmtree(SBX, ignore_errors=True)
print('沙箱清理')

## 深入思考

1. **`Bash(git push*)` 这种 wildcard 真的安全吗？**
   - **半安全**：能用，但**不能写 `Bash(*)`**（放行所有）。**最小粒度原则**：`Bash(gh pr create*)` 比 `Bash(gh *)` 严。**deny 段塞高危子串**：`deny: ['Bash(rm -rf *)', 'Bash(sudo *)']`。
2. **hook 失败会不会把 Claude Code 干挂？**
   - 会。**非零退出码 → 整个 tool 调用报错**。**所以 hook 脚本必须 try/except 包住一切**，**任何异常都返回 0**。
3. **能不能 hook 拒掉一个 tool 调用？**
   - `PreToolUse` 的 hook **可以 exit 2 拒掉**（任何非零 / 2 都会报错）。但**不能动态改 tool_input**。
4. **`/update-config` 这个 Skill 实际就是干这个**：让用户用对话装 Skill 时，**该 Skill 自己**就有 `permissions + hooks` 字段，自动改 settings.json。
5. **多机器同步 settings？**
   - 很多团队把 `~/.claude/settings.json` 和 `~/.claude/skills/` 放在一个 git 仓（dotfiles 模式），用 `chezmoi` / `stow` 同步。

**改一改**：
- 把 `deny` 段加上（把 `rm -rf *` / `sudo` / `git push --force` 拒掉），看 Claude Code 是否会拒
- 给 hook 加一个 exit-2 拒掉逻辑：`PreToolUse` matcher=`Bash` 时，若命令含 `git push --force` 则 exit 2

## 自检 ✅

- [ ] 默写 settings.json 三大段（permissions / hooks / env）
- [ ] 解释 `Bash(gh pr create*)` 与 `Bash(gh *)` 的安全差异
- [ ] 解释「为什么 hook 失败会干挂 Claude Code」
- [ ] 默写 `PreToolUse` 的 exit-2 拒掉语义
- [ ] 解释「为什么 Skill 的 `permissions` 字段是 `install_skill_full` 流程的入口」

## 下一步

→ [`../stage4_专家/38_skill_repo_skeleton.ipynb`](../stage4_专家/38_skill_repo_skeleton.ipynb)